In [1]:
import Fin
from datetime import datetime
import pandas as pd 
import numpy as np
from pathlib import Path
import warnings
import importlib
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
dt                                 = datetime.now().strftime("%d-%m-%Y")
file_path = Path(fr"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN_{dt}")
file_path.mkdir(parents=True, exist_ok=True)

# Consol Model Output

In [2]:
CONSOLIDATED_MODEL_OUTPUT_OLD_path = r"C:\Users\HP\Downloads\New Consol Model Output_2026-09-24.xlsx"
PREV_OT_PATH                       = r"C:\Users\HP\OneDrive - JSW\Desktop\Leads at COB_2026-09-24.xlsx"
SCRUB_PATH                         = r"C:\Users\HP\OneDrive - JSW\Desktop\COB CB Scrub__2592026.xlsx"
OLD_MS                             = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN\Model R 24.09.2026\COB CB Scrub__2492026.xlsx"

CONSOLIDATED_MODEL_OUTPUT_OLD      = Fin.read_file(CONSOLIDATED_MODEL_OUTPUT_OLD_path, sheet_name = 0)
CONSOLIDATED_ADDITIONAL_APPROVALS  = Fin.read_file(CONSOLIDATED_MODEL_OUTPUT_OLD_path, sheet_name = 1)
OT_prev                            = Fin.read_file(PREV_OT_PATH,     sheet_name='Model Output')
SCRUB                              = Fin.read_file(SCRUB_PATH)
OLD_SCRUB                          = Fin.read_file(OLD_MS)

In [3]:
new_cols = list(CONSOLIDATED_MODEL_OUTPUT_OLD.columns)
new_cols[0] = 'Date'
CONSOLIDATED_MODEL_OUTPUT_OLD.columns = new_cols

In [4]:
CONS_MODEL_OT = Fin.cons_model_ot(cmo = CONSOLIDATED_MODEL_OUTPUT_OLD, OT = OT_prev, ms = SCRUB, old_ms= OLD_SCRUB)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

# Model Scrub

In [2]:
scrub_path                         = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN_30-09-2026 m1111 b1\COB CB Scrub__3092026 (1).xlsx"
OD_path                            = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN\NORTH_SOUTH_OD_Update 2.xlsx"
cons_disb_path                     = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN\Disbursement Report_Oct 2022 to Aug 2026.xlsx"
#cons_modelo_path                   = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN"
dt2                                = datetime.now().strftime("%Y-%m-%d")

In [3]:
COB_SCRUB                          = Fin.read_file(scrub_path)
OD_list                            = Fin.read_file(OD_path)
CONSOLIDATED_DISBURSEMENT_REPORT   = Fin.read_file(cons_disb_path)
#CONSOLIDATED_MODEL_OUTPUT          = Fin.read_file(cons_modelo_path, sheet_name = 0)
#CONSOLIDATED_ADDITIONAL_APPROVALS  = Fin.read_file(cons_modelo_path, sheet_name = 1)

FileNotFoundError: File not found: 'C:\Users\HP\OneDrive - JSW\Desktop\COB CB Scrub__2592026.xlsx'

In [ ]:
MODEL_SCRUB, COB_REJECTIONS, ADDITIONAL_APPROVALS, IGLR_MTL_CTL, COMPLETE = Fin.Scrub(ms=COB_SCRUB, cons_disb=CONSOLIDATED_DISBURSEMENT_REPORT, cons_modelo=CONS_MODEL_OT, OD= OD_list, cons_add_app=CONSOLIDATED_ADDITIONAL_APPROVALS)

In [ ]:
Fin.convert_to_excel(file_name = file_path / f"Model_Scrub_Output_B2_{dt2}.xlsx",
    sheets_dict={
    'Model Scrub': MODEL_SCRUB,
    'Rejections': COB_REJECTIONS,
    'Additional Approvals': ADDITIONAL_APPROVALS,
    'Repeat Cycle': IGLR_MTL_CTL,
    'Complete': COMPLETE,
}, hide_sheets=['Complete'])

In [ ]:
Fin.convert_to_excel(
    file_name = file_path / f"New Consol Model Output_B2_{dt2}.xlsx",
    sheets_dict = {
        "OT": CONS_MODEL_OT, "Consol Approvals": ADDITIONAL_APPROVALS[['LEAD ID', 'CB DATE']]})

act = Fin.read_file()

comp = act.merge(COMPLETE, on='LEAD ID', how='outer')
comp[comp['Action_x'] != comp['Action_y']][['Action_x', 'Action_y']].value_counts(dropna = False)

# FOIR

In [2]:
Bureau_path                        = r"C:\Users\HP\OneDrive - JSW\Desktop\BUREAU SPLIT 07OCT2026.xlsx"
Pos_path                           = r"C:\Users\HP\OneDrive - JSW\Desktop\RPT_Portfolio_Outstanding__As On Date-06-10-2026 1 (1).xlsx"
OD_path                            = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN\NORTH_SOUTH_OD_Update 2.xlsx"
inc_path                           = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN\Disbursement Report_Oct 2022 to Aug 2026.xlsx"
#age_path                           = r"C:\Users\Shardul Pimpalkar\Downloads\DOB_05JUN2026.xlsx"

In [3]:
Bureau                             = pd.read_excel(Bureau_path, engine = 'calamine', sheet_name = [0,1,2])
Pos                                = Fin.read_file(Pos_path, usecols = ['CUST ID', 'Loan Product'])
OD                                 = Fin.read_file(OD_path, usecols = ['LEAD ID', 'CUST ID'])
inc                                = Fin.read_file(inc_path)
#age                                = Fin.read_file(age_path)
Pos['CUST ID']                     = pd.to_numeric(Pos['CUST ID'])

In [4]:
Bureau[0]['DOB'] = pd.to_datetime(Bureau[0]['DOB'])
Bureau[0]['AGE'] = (pd.Timestamp.now() - Bureau[0]['DOB']).dt.days // 365

In [5]:
EMI, CV, OT, Las, pdt = Fin.Foir_Model(Bureau, Pos, OD_list= OD, inc=inc)

Processing date: 2026-10-07
CB filter: 244 leads removed (CB expired), 13648 leads retained
CB filtered
Summary filter: 13648 leads after CB date filter
Loan cycle distribution:
LOAN CYCLE
IGL 1    5414
IGL 2    4023
MTL      2989
MTL R    1174
CTL        48
Summary filtered
EMI rows     :  69227, unique leads: 13222
LAS rows     :  69227, unique leads: 13222
CV  rows     :  13648, unique leads: 13648
OT  rows     :  13648, unique leads: 13648
Output Modified
── Final OT Comments distribution ───────────────────────────
Comments
Poor Credit Score                    3125
Overdue&WriteOff                     3034
More than 2 Lenders                  2180
FOIR                                 1747
More than 3 Lenders                  1641
Approve                              1439
Approve After Income Reassessment     310
High Outstanding                      167
Approve- OD                             4
Approve After IC- OD                    1
Total                               13648


In [6]:
#EMI, CV, OT, Las, pdt = Fin.Foir_Model(Bureau, Pos, OD_list=OD, inc=inc, eval_all='Yes')

In [7]:
SCRUB = Fin.read_file(r"C:\Users\HP\OneDrive - JSW\Desktop\COB CB Scrub__7102026 (2).xlsx")

COB_SCRUB = Fin.read_file(r"C:\Users\Shardul Pimpalkar\Downloads\COB CB Scrub__962026.xlsx")

In [8]:
COB_SCRUB = SCRUB.drop_duplicates(subset=['LEAD ID'])
lac = Las.copy()
lac['Product']                     = lac['LEAD_ID'].map(COB_SCRUB.set_index('LEAD ID')['LOAN CYCLE'])
lac['Cust ID']                     = lac['LEAD_ID'].map(COB_SCRUB.set_index('LEAD ID')['Customer ID'])
lac['State']                       = lac['LEAD_ID'].map(COB_SCRUB.set_index('LEAD ID')['State'])
lac['Cust Name']                   = lac['LEAD_ID'].map(COB_SCRUB.set_index('LEAD ID')['Customer Name'])
lac['Branch']                      = lac['LEAD_ID'].map(COB_SCRUB.set_index('LEAD ID')['Branch Name'])

lac_order =['LEAD_ID', 'Cust ID', 'Cust Name', 'State', 'Branch', 'Product','Name of the Institution', 'Date Opened', 'Sanction Amount',
       'Current Balance', 'Final EMI', 'Include in FOIR', 'Remarks', 'Equifax ID', 'Source', 'Date Reported', 'Joint Account Identifier',
       'Relation', 'Term freq.', 'ACCOUNT_STATUS', 'Loan_Category']
LEADS_AT_COB = lac[lac_order].copy()

In [9]:
#Fin.convert_to_excel(file_name = file_path / f"Leads at COB_2_{pdt}.xlsx",
                     #sheets_dict = {
    #"Approvals": OT[OT['Comments'].str.contains('Approve')],
    #"FOIR": OT[OT['Comments'] == 'FOIR'],
    #"Loanwise Adjustment Sheet": Las, 'Model Output': OT}, hide_sheets = ['Model Output'])

In [10]:
_ot_no_rr = OT.drop(columns=['Rejection Reasons'], errors='ignore')
Fin.convert_to_excel(file_name = file_path / f"Leads at COB2_{pdt}.xlsx", 
                     sheets_dict = {
    "Approvals": _ot_no_rr[_ot_no_rr['Comments'].str.contains('Approve')], 
    "FOIR": _ot_no_rr[_ot_no_rr['Comments'] == 'FOIR'],
    "Loanwise Adjustment Sheet": LEADS_AT_COB, 'Model Output': OT}, hide_sheets = ['Model Output'])

In [ ]:
#pcs_od_approval sheet

In [12]:
Fin.export_psc_od_approvals(OT, file_path / f"PSC_OD_Approvals_{pdt}.xlsx")

# Back- End Approvals

In [2]:
demographic_path                    = r"C:\Users\HP\OneDrive - JSW\Desktop\Demographic_Data_30SEP2026.xlsx"
ot_path                             = r"C:\Users\HP\OneDrive - JSW\Desktop\Model Output Backend _30-09-2026.xlsx"
scrub_path                          = r"C:\Users\HP\OneDrive - JSW\Desktop\COB CB Scrub__3092026 (2).xlsx"
inc_path                            = r"C:\Users\HP\OneDrive - JSW\Desktop\MODEL RUN\Disbursement Report_Oct 2022 to Aug 2026.xlsx"
dt                                  = datetime.now().strftime("%Y-%m-%d")

In [3]:
DEMOGRAPHIC                         = Fin.read_file(demographic_path)
COB_SCRUB                           = Fin.read_file(scrub_path)
COB_SCRUB                           = COB_SCRUB.drop_duplicates(subset=['LEAD ID'])
inc                                 = Fin.read_file(inc_path)

In [4]:
OT                                  = Fin.read_file(r"C:\Users\HP\OneDrive - JSW\Desktop\Model Output Backend _30-09-2026.xlsx", sheet_name = 'Model Output')

In [5]:
def Back_End_Apps(demog, OT, cams, cdl):
    """
    Process backend-approval leads: KYC cross-checks + income reassessment.

    Backend approvals are required for leads whose CAMS data was submitted
    today AND whose demographic data (Voter ID, Aadhaar, IFSC, Account No)
    passes a four-way consistency check against the demographics master.

    Steps:
      1. Split demog into 'blanks' (no account status) and the filled records.
      2. De-duplicate filled records, keeping the first non-blank per customer.
      3. Merge blanks with filled records to get canonical KYC values.
      4. Perform four boolean checks (Voter, Aadhaar, IFSC, Account).
      5. Map model outputs (Credit Score, MFI Outstanding, Lender Count, Comments).
      6. Filter to approvable Comments only.
      7. Verify CAMS was submitted today; split into CAMS_NOT_FILLED / apps2.
      8. Compute FOIR from COB installment / COB income.
      9. Run income-reassessment logic to derive New Income.
     10. Assign BA Comment using priority-ordered np.select conditions.

    Parameters
    ----------
    demog              : Demographics master DataFrame.
    OT                 : Model output table (OT sheet from Foir_Model or Joint).
    cams               : CAMS submission sheet — provides CAMS Submission Date, COB Income, COB Installment.
    cdl                : Consolidated disbursement list — previous income and disbursement date.

    Returns
    -------
    ALL_TRUE           : Leads where all four KYC checks pass, with BA Comments.
    FALSE              : Leads where at least one KYC check fails.
    CAMS_NOT_FILLED    : Approved leads where CAMS was NOT submitted today.
    """
    # ── Step 1: Split on blank account status ─────────────────────────────────
    # Leads with a space (' ') in ACCOUNT STATUS are the 'current' records;
    # all others are historical / closed account records used for KYC reference.
    blanks = demog[demog['ACCOUNT STATUS'] == " "].copy()
    demog  = (
        demog[demog['ACCOUNT STATUS'] != " "]
        .sort_values('ACCOUNT STATUS')
        .drop_duplicates(keep='first', subset='CUSTOMER ID')
        .copy()
    )
    demog = demog[['CUSTOMER ID', 'CUSTOMER VOTER ID', 'CUSTOMER AADHAR NO', 'IFSC_CODE', 'AC_NO']].copy()
    demog = demog.rename(columns={
        'CUSTOMER VOTER ID': 'Voter ID',
        'CUSTOMER AADHAR NO': 'Aadhaar No',
        'IFSC_CODE': 'IFSC',
        'AC_NO': 'Account No',
    })

    # ── Step 2: Merge and perform KYC checks ──────────────────────────────────
    blanks = blanks.loc[:, ~blanks.columns.duplicated()].copy()
    blanks = blanks.merge(demog, how='left', on='CUSTOMER ID')
    blanks['Voter Check']   = blanks['CUSTOMER VOTER ID'] == blanks['Voter ID']
    blanks['Aadhaar Check'] = blanks['CUSTOMER AADHAR NO'] == blanks['Aadhaar No']
    blanks['IFSC Check']    = blanks['IFSC_CODE'] == blanks['IFSC']
    blanks['Account Check'] = blanks['Account No'] == blanks['AC_NO']

    # ── Step 3: Map model outputs onto blanks ─────────────────────────────────
    blanks['Credit Score']    = blanks['LEAD ID'].map(OT.set_index('Lead ID')['Credit Score'])
    blanks['MFI Outstanding'] = blanks['LEAD ID'].map(OT.set_index('Lead ID')['MFI Outstanding'])
    blanks['Lender Count']    = blanks['LEAD ID'].map(OT.set_index('Lead ID')['Lender Count'])
    blanks['Comments']        = blanks['LEAD ID'].map(OT.set_index('Lead ID')['Comments'])

    #non_apps = blanks[~blanks['Comments'].isin(['Approve After Income Reassessment', 'Approve', 'FOIR', 'Approve After IC-OD', 'Approve -OD'])].copy()  Approve AFTER IC - OD

    # ── Step 4: Filter to approvable Comments ─────────────────────────────────
    apps = blanks[
        blanks['Comments'].isin([
            'Approve After Income Reassessment', 'Approve',
            'Approve AFTER IC - OD', 'Approve -OD', 'FOIR',
            'Approve AFTER IC - MT', 'Approve AFTER IC - OD ',
        ])
    ].copy()

    # ── Step 5: Map COB sheet values ──────────────────────────────────────────
    apps['CAMS Submission Date'] = apps['LEAD ID'].map(cams.set_index('LEAD ID')['CAMS Submission Date'])
    apps['COB Income']           = apps['LEAD ID'].map(cams.set_index('LEAD ID')['Household Total Monthly Income'])
    apps['COB Installment']      = apps['LEAD ID'].map(cams.set_index('LEAD ID')['Revised Monthly Installment'])

    cdl['CUST ID'] = pd.to_numeric(cdl['CUST ID'])
    cdl = cdl.drop_duplicates(subset='CUST ID', keep='first')
    apps['Old Income'] = apps['CUSTOMER ID'].map(cdl.set_index('CUST ID')['Total Monthly Income'])
    apps['DD']         = apps['CUSTOMER ID'].map(cdl.set_index('CUST ID')['Disbursement Date'])

    apps_ret = apps.copy()

    # ── Step 6: Split on today's CAMS submission ──────────────────────────────
    today = pd.Timestamp.now().date()
    #CAMS_NOT_FILLED = apps[(apps['CAMS Submission Date'].isna()) | (apps['CAMS Submission Date'] == 0)].copy()
    apps["CAMS Submission Date"] = pd.to_datetime(
        apps["CAMS Submission Date"], format="%d-%m-%Y", errors="coerce"
    )
    #apps2 = apps[(~apps['CAMS Submission Date'].isna()) & (apps['CAMS Submission Date'] != 0)].copy()
    CAMS_NOT_FILLED = apps[apps["CAMS Submission Date"].dt.date != today].copy()
    CAMS_NOT_FILLED['BA Comment'] = 'CAMS Not Filled'
    apps2           = apps[apps["CAMS Submission Date"].dt.date == today].copy()

    # ── Step 7: Compute FOIR ──────────────────────────────────────────────────
    apps2['FOIR'] = apps2['COB Installment'] / apps2['COB Income']
    
    # ── Step 8: KYC four-way split ─────────────────────────────────────────────
    cols_check = ['Voter Check', 'Aadhaar Check', 'Account Check', 'IFSC Check']
    mask = apps2[cols_check].all(axis=1)

    ALL_TRUE = apps2[mask].copy().reset_index(drop=True)   # All KYC checks passed
    FALSE    = apps2[~mask].copy().reset_index(drop=True)  # At least one check failed
    FALSE['BA Comment'] = 'KYC/ Bank Issue'
    at_return = ALL_TRUE.copy()

    # ── Step 9: Income reassessment (IC) ──────────────────────────────────────
    # Same logic as in Scrub(): compute New Income from COB Installment / 0.47
    # when FOIR exceeds 50 %.  Then take the max of (New Income, Old Income, COB Income).
    foir_meets = ALL_TRUE['FOIR'] <= 0.5
    mask = ALL_TRUE['FOIR'] > 0.5
    ALL_TRUE['COB Installment'] = pd.to_numeric(ALL_TRUE['COB Installment'], errors='coerce')
    ALL_TRUE['New Income'] = np.nan
    
    ALL_TRUE.loc[mask, 'New Income'] = (ALL_TRUE.loc[mask, 'COB Installment'].values / 0.47).round(-2)

    # Cap at ₹25 000 if installment does not justify the NQA bracket
    ALL_TRUE.loc[
        (ALL_TRUE['New Income'] > 25_000) & (ALL_TRUE['COB Installment'] <= 12_500),
        'New Income'
    ] = 25_000
    
    ALL_TRUE.loc[
        (ALL_TRUE['New Income'] > 75_000) & (ALL_TRUE['COB Installment'] <= 36_500),
        'New Income'
    ] = 75_000

    # Take max of back-calculated new income vs. previous disbursed income vs. COB income
    for col in ['New Income', 'Old Income', 'COB Income']:
        ALL_TRUE[col] = pd.to_numeric(ALL_TRUE[col], errors='coerce')
    has_old = ALL_TRUE['Old Income'].notna()
    ALL_TRUE.loc[has_old, 'New Income'] = (
        ALL_TRUE.loc[has_old, ['New Income', 'Old Income', 'COB Income']].max(axis=1)
    )

    # ── Step 10: Income bracket masks ─────────────────────────────────────────
    old_nqa    = ALL_TRUE['Old Income'].between(25_000, 75_000, inclusive='right')
    old_qa     = (ALL_TRUE['Old Income'] <= 25_000) | (ALL_TRUE['Old Income'].isna())
    new_qa     = (
        (ALL_TRUE['New Income'] <= 25_000) |
        ((ALL_TRUE['New Income'].isna()) & (ALL_TRUE['COB Income'] <= 25_000))
    )
    new_nqa    = (
        ALL_TRUE['New Income'].between(25_000, 75_000, inclusive='right') |
        ((ALL_TRUE['New Income'].isna()) & ALL_TRUE['COB Income'].between(25_000, 75_000, inclusive='right'))
    )
    inc_change = ALL_TRUE['New Income'] != ALL_TRUE['COB Income']  # COB vs New income differs

    today    = datetime.now()
    date_1yr = today - timedelta(days=365)  # keep as datetime
    ALL_TRUE['DD'] = pd.to_datetime(ALL_TRUE['DD'])

    # ── Step 11: BA Comment assignment ────────────────────────────────────────
    # Priority-ordered conditions that mirror the Scrub() hold / IC logic,
    # but applied at the backend-approval stage with full KYC and CAMS data.
    bac_condn2 = (
        (ALL_TRUE['New Income'] > 75_000) | (ALL_TRUE['Old Income'] > 75_000),  # On Hold (High Income)
        old_qa & new_nqa & ALL_TRUE['Loan Type'].isin(['Mid Loan']),             # On hold (MTL NQA)
        old_qa & new_nqa & (ALL_TRUE['DD'] > date_1yr),                         # On hold (Recent QA)
        old_qa & new_nqa & inc_change,                                           # NQA- IC
        old_qa & new_nqa,                                                        # Approve
        old_nqa & new_qa,                                                        # QA- IC
        old_qa  & new_qa & inc_change,                                           # QA- IC
        old_qa  & new_qa,                                                        # Approve
        old_nqa & new_nqa & inc_change,                                          # NQA- IC
        old_nqa & new_nqa,                                                       # Approve
    )
    bac_value2 = (
        'On hold (MTL NQA)', 'On hold (Recent QA)', 'On Hold (High Income)',
        'NQA- IC', 'Approve',
        'QA- IC',
        'QA- IC', 'Approve',
        'NQA- IC', 'Approve',
    )
    ALL_TRUE['BA Comment'] = np.select(bac_condn2, bac_value2, default='Approve')

    # ── Step 12: Special case — MTL NQA hold reclassification ─────────────────
    # MTL leads whose computed New Income exceeds ₹75 000 but instalment
    # is within NQA bounds → cap income at ₹75 000 and assign NQA-IC.
    mask = (
        (ALL_TRUE['New Income'] > 75_000) &
        (ALL_TRUE['COB Installment'] <= 365_000) &
        (
            (ALL_TRUE['BA Comment'] == 'On hold (MTL NQA)') |
            (
                (ALL_TRUE['BA Comment'] == 'On Hold (High Income)') &
                (ALL_TRUE['Loan Type'] == 'MTL') &
                (ALL_TRUE['Old Income'] < 25_000)
            )
        )
    )
    ALL_TRUE.loc[mask, 'New Income']    = 75_000
    ALL_TRUE.loc[mask, 'BA Comment']    = 'NQA- IC'

    # ── Output summary ─────────────────────────────────────────────────────────
    print("── BA Comment distribution (ALL_TRUE) ───────────────────────")
    print(ALL_TRUE['BA Comment'].value_counts().to_string())
    total_all = len(ALL_TRUE)
    total_false = len(FALSE)
    total_cams_not_filled = len(CAMS_NOT_FILLED)
    print(f"{'ALL_TRUE (all KYC pass)':<40} {total_all:>6}")
    print(f"{'FALSE (KYC fail)':<40} {total_false:>6}")
    print(f"{'CAMS_NOT_FILLED':<40} {total_cams_not_filled:>6}")
    blank = pd.concat([ALL_TRUE, FALSE, CAMS_NOT_FILLED], ignore_index=True)

    return ALL_TRUE, FALSE, CAMS_NOT_FILLED, blank

In [6]:
print(list(DEMOGRAPHIC.columns))

['STATE', 'VILLAGENAME', 'BRANCH_NAME', 'CENTER NAME', 'LEAD ID', 'CUSTOMER ID', 'ACCOUNT STATUS', 'LEAD CURREN STAGE', 'Loan Type', 'PRIMARY CUSTOMER NAME', 'CUST DOB', 'CUSTOMER AADHAR NO', 'CUSTOMER VOTER ID', 'CUSTOMER PAN NO', 'CUST MOTHER NAME', 'CUST FATHER NAME', 'SPOUSE NAME', 'SPOUSE DOB', 'CO APPLICANT NAME', 'CO APP DOB', 'CO-APPICANT AADHAR NO', 'CO APP VOTER ID', 'CO APP MOTHER NAME', 'CO APP FATHER NAME', 'RELATION WITH CO-APPLICANT', 'IFSC_CODE', 'BANK_NAME', 'BRANCH_NAME', 'AC_HOLDER_NAME', 'AC_NO', 'TOTAL MONTHLY INCOME', 'CUST RESDIANTAL PROOF', 'COMMENTS']


In [7]:
ALL_TRUE, FALSE, CAMS_NOT_FILLED, COMPLETE = Fin.Back_End_Apps(demog=DEMOGRAPHIC, cams=COB_SCRUB, cdl=inc, OT=OT)

── BA Comment distribution (ALL_TRUE) ───────────────────────
BA Comment
Approve                  440
On hold (MTL NQA)        142
QA- IC                    67
NQA- IC                   44
On hold (Recent QA)       14
On Hold (High Income)      7
COB Income NaN: 0 / 714
New Income NaN:  10 / 714
new_qa True:     382
new_nqa True:    325
ALL_TRUE (all KYC pass)                     714
FALSE (KYC fail)                             83
CAMS_NOT_FILLED                              76


In [8]:
sheets_dict={
    "All True": ALL_TRUE, 
    "Cams not filled": CAMS_NOT_FILLED, 
    'False': FALSE,
    'Complete': COMPLETE}

Fin.convert_to_excel(file_name = file_path / f"Back End Approvals{dt}.xlsx", sheets_dict=sheets_dict, hide_sheets=['Complete'])

with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    for sheet_name, df in sheets_dict.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

act = pd.read_excel(r"C:\Users\Shardul Pimpalkar\Downloads\Blanks 34.xlsx", skiprows = 0)
comp = act.merge(at, on='LEAD ID', how='outer')
comp[['Comments_x', 'BA Comment']].value_counts(dropna = False)